In [2]:
import os
proxy_url = "http://127.0.0.1:7890" 
os.environ["http_proxy"] = proxy_url
os.environ["https_proxy"] = proxy_url
import sys
import requests
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments, AllChem, rdMolDescriptors

In [3]:
# smiles规范化-----get_canonical
def get_canonical(smi):
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                can_smi = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
                print(f"规范化结果: {can_smi}")
            else:
                print("错误: 无法解析该 SMILES，请检查输入是否正确。")
        except Exception as e:
            print(f"程序运行出错: {e}")

In [4]:
# 提取PubChem的分子描述-----get_pubchem_description
def get_pubchem_description(smiles):
    """
    根据用户提供的 JSON 结构，精准提取 Names and Identifiers 下的 Record Description
    """
    try:
        # 第一步：获取 CID
        compounds = pcp.get_compounds(smiles, namespace='smiles')
        if not compounds: return "CID not found."
        cid = compounds[0].cid
        
        # 第二步：获取 PUG View JSON
        pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON"
        response = requests.get(pug_url, timeout=15)
        if response.status_code != 200: return "API Error."
        
        data = response.json()
        final_descriptions = []

        # 第三步：按照目标路径精准导航
        # 路径: Record -> Section
        sections = data.get("Record", {}).get("Section", [])
        
        for sec in sections:
            # 找到一级目录: Names and Identifiers
            if sec.get("TOCHeading") == "Names and Identifiers":
                sub_sections = sec.get("Section", [])
                
                for sub_sec in sub_sections:
                    # 找到二级目录: Record Description
                    if sub_sec.get("TOCHeading") == "Record Description":
                        informations = sub_sec.get("Information", [])
                        
                        for info in informations:
                            # 提取 StringWithMarkup 里的所有文本
                            value = info.get("Value", {})
                            markup_list = value.get("StringWithMarkup", [])
                            
                            for markup in markup_list:
                                text = markup.get("String", "")
                                if text:
                                    final_descriptions.append(text.strip())

        # 第四步：拼接所有找到的段落
        if not final_descriptions:
            return ""
            
        return "".join(final_descriptions)

    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# 提取RDKit的分子描述----get_rdkit_description
def get_rdkit_description(smiles):
    # 1. 分子对象创建与合法性检查
    mol = Chem.MolFromSmiles(smiles) 
    if not mol:  
        return "Error: Invalid SMILES string."  # 返回错误提示并终止

    # 为了确保输出的描述是针对标准结构的，进行规范化处理
    canonical_smi = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)  # 生成标准且含手性的 SMILES
    mol = Chem.MolFromSmiles(canonical_smi)  # 重新加载标准分子对象以保证计算精度

    # 2. 基础物理常数提取
    Formula = rdMolDescriptors.CalcMolFormula(mol)  # 计算分子的化学式（如 C27H33N3O2）
    MolWt = round(Descriptors.MolWt(mol), 2)  # 计算精确分子量，保留两位小数
    MolLogP = round(Descriptors.MolLogP(mol), 2)  # 计算脂水分配系数 LogP（反映亲脂性）
    TPSA = round(Descriptors.TPSA(mol), 2)  # 计算总极性表面积 TPSA（反映渗透性）

    # 3. 结构特征与原子统计
    NumHDonors = Descriptors.NumHDonors(mol)  # 统计氢键供体数量
    NumHAcceptors = Descriptors.NumHAcceptors(mol)  # 统计氢键受体数量
    RingCount = mol.GetRingInfo().NumRings()  # 统计分子中环的总数
    NumRotatableBonds = Descriptors.NumRotatableBonds(mol)  # 统计可旋转化学键数量（反映分子柔性）
    stereo_centers = rdMolDescriptors.CalcNumAtomStereoCenters(mol)  # 统计手性中心（立体中心）的数量
    HeavyAtoms = mol.GetNumHeavyAtoms()
    # 获取芳香环数量 (NumAromaticRings)
    NumAromaticRings = Descriptors.NumAromaticRings(mol)
    
    # 获取 sp3 杂化碳原子比例 (FractionCSP3)
    # 这个值反映分子的立体程度，使用 round 保留两位小数
    FractionCSP3 = round(Descriptors.FractionCSP3(mol), 2)
    
    # 获取 Balaban J 指数 (BalabanJ)
    # 这是一个描述分子拓扑复杂性的指数
    BalabanJ = round(Descriptors.BalabanJ(mol), 2)

    # 4. 关键官能团扫描（使用 RDKit 内置的片段库）
    fg_list = []  # 创建一个列表用于存放检测到的官能团
    if Fragments.fr_benzene(mol) > 0: fg_list.append("aromatic benzene rings")  # 检测苯环
    if Fragments.fr_amide(mol) > 0: fg_list.append("amide groups")  # 检测酰胺键
    if Fragments.fr_Ar_OH(mol) > 0: fg_list.append("phenolic hydroxyls")  # 检测酚羟基
    if Fragments.fr_NH2(mol) > 0: fg_list.append("primary amine groups")  # 检测伯胺
    if Fragments.fr_halogen(mol) > 0: fg_list.append("halogen substituents")  # 检测卤素（F, Cl, Br, I）
    if Fragments.fr_ester(mol) > 0: fg_list.append("ester linkages")  # 检测酯基
    if Fragments.fr_ether(mol) > 0: fg_list.append("ether groups")  # 检测醚键
    
    # 构造官能团描述短语
    if fg_list:  # 如果找到了已知官能团
        fg_text = "The structural framework incorporates " + ", ".join(fg_list) + ". "  # 拼接官能团描述词
    else:  # 如果没找到常见官能团
        fg_text = "The structure consists of a complex hydrocarbon-based framework. "  # 使用通用描述

    # 5. 最终描述文本生成
    description = (
    f"{smiles}：This molecule, defined by the chemical formula {Formula}, possesses a molecular weight of {MolWt} g/mol. "
    f"{fg_text}"
    f"Topological analysis indicates the presence of {RingCount} ring system(s), including {NumAromaticRings} aromatic ring(s), "
    f"with a connectivity complexity quantified by a Balaban J index of {BalabanJ}. "
    f"The stereochemical configuration is defined by {stereo_centers} stereogenic center(s)). "
    f"From a physicochemical perspective, the molecule exhibits a LogP of {MolLogP} and a TPSA of {TPSA} Å², key parameters governing its lipophilicity and polar surface interactions. "
    f"It contains {HeavyAtoms} heavy atom(s), reflecting the size of its non-hydrogen atomic framework. "
    f"The molecular flexibility is moderated by {NumRotatableBonds} rotatable bonds and a FractionCSP3 of {FractionCSP3}, "
    f"while its interaction profile is defined by {NumHDonors} hydrogen bond donor(s) and {NumHAcceptors} acceptor(s)."
)
    return description  # 返回最终生成的知识文本

In [5]:
# 纯净的理化性质----get_all_rdkit_properties
def get_all_rdkit_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    # 1. 基础识别
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mw = round(Descriptors.MolWt(mol), 4)
    exact_mw = round(Descriptors.ExactMolWt(mol), 4)
    
    # 2. 理化性质
    logp = round(Descriptors.MolLogP(mol), 4)
    tpsa = round(Descriptors.TPSA(mol), 4)
    
    # 3. 结构计数
    heavy_atoms = mol.GetNumHeavyAtoms()
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    rot_bonds = Descriptors.NumRotatableBonds(mol)
    
    # 4. 环与拓扑
    rings = rdMolDescriptors.CalcNumRings(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    stereo_centers = rdMolDescriptors.CalcNumAtomStereoCenters(mol)

    return {
        "Formula": formula, "MolWt": mw, "ExactMolWt": exact_mw,
        "LogP": logp, "TPSA": tpsa, "Heavy_Atoms": heavy_atoms,
        "H_Bond_Donors": hbd, "H_Bond_Acceptors": hba, "Rotatable_Bonds": rot_bonds,
        "Ring_Count": rings, "Aromatic_Rings": aromatic_rings,
        "Stereocenters": stereo_centers
    }

In [ ]:
if __name__ == "__main__":
    smi = "COC1=CC(=CC(=C1)C#CC2=NN(C3=NC=NC(=C23)N)C4CCN(C4)C(=O)C=C)OC"

    rdkit = get_rdkit_description(smi)
    pubchem = get_pubchem_description(smi)
    res = res = rdkit + "\n" + pubchem
    print(res)

CCCCCOC(=O)Nc1c(cn(c(=O)n1)[C@H]2[C@@H]([C@@H]([C@H](O2)C)O)O)F：This molecule, defined by the chemical formula C15H22FN3O6, possesses a molecular weight of 359.35 g/mol. The structural framework incorporates amide groups, halogen substituents, ether groups. Topological analysis indicates the presence of 2 ring system(s), including 1 aromatic ring(s), with a connectivity complexity quantified by a Balaban J index of 2.07. It contains 4 stereogenic center(s). From a physicochemical perspective, the molecule exhibits a LogP of 0.76 and a TPSA of 122.91 Å². It contains 25 heavy atom(s), reflecting the size of its non-hydrogen atomic framework. The molecular flexibility is moderated by 6 rotatable bonds and a FractionCSP3 of 0.67, while its interaction profile is defined by 3 hydrogen bond donor(s) and 8 acceptor(s).
Capecitabine is a carbamate ester that is cytidine in which the hydrogen at position 5 is replaced by fluorine and in which the amino group attached to position 4 is converted 

In [12]:
import os
import glob
import pandas as pd

# 你已实现的函数
# def get_rdkit_description(smi): ...
# def get_pubchem_description(smi): ...

def build_description(smi):
    rdkit = get_rdkit_description(smi)
    pubchem = get_pubchem_description(smi)

    # 失败判定（你可以按自己的返回格式调整）
    rdkit_ok = bool(rdkit and str(rdkit).strip())
    pubchem_ok = bool(pubchem and str(pubchem).strip())

    if rdkit_ok and pubchem_ok:
        return rdkit + "\n" + pubchem, None
    if rdkit_ok and not pubchem_ok:
        return rdkit, "pubchem_failed"
    if pubchem_ok and not rdkit_ok:
        return pubchem, "rdkit_failed"
    return "", "rdkit_failed+pubchem_failed"

def process_csv(path):
    df = pd.read_csv(path)
    if "smiles" not in df.columns:
        print(f"跳过（无 smiles 列）: {path}")
        return

    failures = []
    descriptions = []

    for smi in df["smiles"].astype(str):
        desc, fail = build_description(smi)
        descriptions.append(desc)
        if fail:
            failures.append({"smiles": smi, "reason": fail})

    df["description"] = descriptions

    out_path = path.replace(".csv", "_with_desc.csv")
    df.to_csv(out_path, index=False)
    print(f"完成: {out_path}")

    if failures:
        fail_path = path.replace(".csv", "_desc_failures.csv")
        pd.DataFrame(failures).to_csv(fail_path, index=False)
        print(f"失败记录: {fail_path} ({len(failures)} 条)")

def collect_csv_files(input_path):
    if os.path.isdir(input_path):
        return glob.glob(os.path.join(input_path, "**/*.csv"), recursive=True)
    if os.path.isfile(input_path) and input_path.lower().endswith(".csv"):
        return [input_path]
    raise ValueError(f"无效输入路径: {input_path}")

def main(input_path):
    files = collect_csv_files(input_path)
    for path in files:
        process_csv(path)

if __name__ == "__main__":
    # 改成你要处理的路径：可以是文件夹，也可以是单个 CSV
    input_path = "/home/fsy23/UniPoly/moleculenet/regression/Lipophilicity.csv"
    main(input_path)


完成: /home/fsy23/UniPoly/moleculenet/regression/Lipophilicity_with_desc.csv
失败记录: /home/fsy23/UniPoly/moleculenet/regression/Lipophilicity_desc_failures.csv (3320 条)


In [14]:
import json
import pandas as pd

def csv_to_jsondict(input_csv, output_json, smiles_col="smiles", desc_col="description"):
    df = pd.read_csv(input_csv)
    if smiles_col not in df.columns or desc_col not in df.columns:
        raise ValueError(f"缺少列: {smiles_col} 或 {desc_col}")

    data = {}
    for _, row in df.iterrows():
        smi = str(row[smiles_col])
        desc = str(row[desc_col]) if pd.notna(row[desc_col]) else ""
        data[smi] = desc

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Done: {output_json} (items={len(data)})")

if __name__ == "__main__":
    csv_to_jsondict(
        input_csv="/home/fsy23/UniPoly/moleculenet/regression/Lipophilicity_with_desc.csv",
        output_json="moleculenet/regression/Lipo_smiles_desc.json"
    )


Done: moleculenet/regression/Lipo_smiles_desc.json (items=4200)
